# Implementation of a transformer model
In this exercise we will implement the transformer model (an architecture based on LLama2). The squeleton of the code is provided and you should complete it (mostly self attention mechanism).

In [1]:
import time
from time import gmtime, strftime
import math
import shutil
import os
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer

import re
import random
from collections import Counter

SEED = 42

c:\Users\jdorn\Documents\M2\NLP Today\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

class RoPE:
    @staticmethod
    def get_cos_sin_(
        embedding_size: int,
        precomp_len: int = 4096,
        theta_param: float = 10000,
        factor: float = 1
    ):
        theta = factor / (
              theta_param ** (torch.arange(0, embedding_size, 2).float() / embedding_size)
        )
        seq_idx = torch.arange(
            0, precomp_len, 1.0
        )
        rotation_matrix = seq_idx.unsqueeze(1)@theta.unsqueeze(0)

        return (
            rotation_matrix.cos().unsqueeze(-1).repeat(1,1 ,2).view(precomp_len, -1,1).squeeze(),
            (rotation_matrix.sin().unsqueeze(-1).repeat(1,1 ,2) * torch.Tensor([-1,1])).view(precomp_len, -1, 1).squeeze()
        )

    def __init__(
        self,
        embedding_size: int,
        precomp_len: int = 4096,
        theta_param: float = 10000,
        factor: float = 1
    ):
        assert embedding_size%2 == 0, "RoPE is only implemented for dimension being multiple of 2"

        self.embedding_size = embedding_size
        self.theta_param = theta_param
        self.factor = factor

        self.rope_cos, self.rope_sin =\
            RoPE.get_cos_sin_(
                self.embedding_size,
                precomp_len,
                self.theta_param,
                self.factor
            )

    def __call__(self, x: torch.Tensor, start_pos_id: int  = 0):
        '''
            Apply RoPE embedding to a tensor x

            Parameters
            x : [torch.Tensor]
                Expected dim BxHxLxN
            start_pos_id : int
                The start of the sequence to apply RoPE
        '''
        B, H, L, N = x.shape # batch size, sequence len, embed dim
        if L > self.rope_cos.shape[0]:
            # The precomputed RoPE has a too small size
            self.rope_cos, self.rope_sin =\
                RoPE.get_cos_sin_(
                    self.embedding_size,
                    L,
                    self.theta_param,
                    self.factor
                )
        if (x.device != self.rope_cos.device):
            # if precomputed tensor and rope are not on the same device
            self.rope_cos = self.rope_cos.to(x.device)
            self.rope_sin = self.rope_sin.to(x.device)

        return ( x * self.rope_cos[start_pos_id: start_pos_id + L]
                 +
                 x.reshape(B, H, L, N//2, 2).flip(-1).view(B, H, L, N) * self.rope_sin[start_pos_id: start_pos_id + L]
               )

## Self Attention with no mask

Let consider $Q, K, V \in R^{B\times H \times L \times N}$ for each batch element (B) for each heads (H) we want to compute $$A = \frac{Q_{bh}.K_{bh}^{t}}{\sqrt{N}}$$

Attention weights are defined by $$
softmax(A_i) = \begin{pmatrix}
\frac{e^{A_{i1}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} &
\frac{e^{A_{i2}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} &
\dots &
\frac{e^{A_{iL}}}{\sum\limits_{j=1}^L  e^{A_{ij}}}
\end{pmatrix}$$
And $$
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} &
\frac{e^{A_{12}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} &
\dots &
\frac{e^{A_{1L}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} &
\frac{e^{A_{22}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} &
\dots &
\frac{e^{A_{2l}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} &
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} &
\dots &
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}}
\end{pmatrix}$$


Notice that the function softmax can be used to compute this matrix [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html#torch.softmax).


## Masked Attention
In many cases, certains tokens will not compute attention with later tokens (padding or decoding approaches). Typically in decoder only architectures we would have the attention weights matrix only considering lower diagonal values :

$$
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^1  e^{A_{1j}}} &
0 &
\dots & 0 &
0 \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} &
\frac{e^{A_{22}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} &
\dots & 0 &
0 \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{{L-1}1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} &
\frac{e^{A_{{L-1}2}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} &
\dots &
\frac{e^{A_{L-1L-1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}}& 0 \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} &
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} &
\dots &
\frac{e^{A_{LL-1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} &
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} \\
\end{pmatrix}$$

A simple implementation would consist in setting the upper diagonal of A to $-\infty$ and apply the softmax. We will thus consider a matrix name $M$ (or `mask` in the code) having upper diagonal element to $-\infty$ :
$$ M =
\begin{pmatrix}
0&
-\infty &
\dots & -\infty&
-\infty \\
0 &
0 &
\dots & -\infty &
-\infty \\
\vdots & \vdots & \ddots & \vdots \\
0 &
0 &
\dots &
0& -\infty \\
0 &
0 &
\dots &
0 &
0\\
\end{pmatrix}
$$

And apply softmax on $A + M$

In [3]:
def multi_head_attention(
    q : torch.FloatTensor,
    k : torch.FloatTensor,
    v : torch.FloatTensor,
    mask : torch.FloatTensor
):
    """
    q: B x H x Q x N, k and v: B x H x K x N
    mask: B x Q x K with 0 (keep) or -inf (masked)
    Returns the attention weights and the attention output
    """
    N = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(N)   # B x H x Q x K
    if mask is not None:
        scores = scores + mask.unsqueeze(1)            # same mask for every head
    attention_weights = torch.softmax(scores, dim=-1)
    attention_output = attention_weights @ v           # B x H x Q x N
    return attention_weights, attention_output


In [4]:
q, k, v = (torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4))
mask = torch.triu(-q.new_ones(1, 3, 3) *  torch.inf, diagonal =  1)
attention_weigths, attention_output = multi_head_attention(q, k, v, mask)

assert(torch.all(attention_weigths == torch.Tensor([
          [[[1., 0., 0.],
            [1/2, 1/2, 0.],
            [1/3, 1/3, 1/3]]]])))

##  Create the attention module

In pytorch module are block of neural networks that inherit from the class `nn.Module`.  In the following you willbe asked to complete the attention code. The attention module compute the Query $Q$, the Keys $K$ and Values $V$ for each heads and compute attention. The forward methods is the method implementing these operations. In this lab we use a mult-head attention, meaning that we compute attention for each heads, that can be done with a for loop or using batched operation (it is up to you). Also as we follow the Llama 2 architecture, we use for positional embeding the RoPe positonal encoding that is applied on the two matrix $Q$ and $K$. The forward function should compute for an input $X \in \mathbb{R}^{L \times N}$ (in the decoder only case) the following:

1. $Q_h = RoPE(W^q_{h}X^{\intercal})$ (for all heads h) with $W^q_h \in \mathbb{R}^{N \times N}$
2. $K_h = RoPE(W^k_{h}X^{\intercal})$ (for all heads h)
3. $V_h = W^v_{h}X^{\intercal}$ (for all heads h)
4. $A_h =  Attention(Q_h, K_h, V_h)$ (for all heads h)
5. $A$ = [A_1, A_2, \dots A_H] (the concatenation along the last dimension)
6. $O = W^oA^{\intercal}$ with $w^{o} \in \mathbb{R}^{N \times NH}$ (NH being the scalar multplication of N and H)

Notice that Q, K and V can be computed without a loop using $W_{q}$ (respectively for $W_k$ and $W_v$) .


In [5]:
class RoPEAttentionModule(nn.Module):
    """ A cross/self attention pytorch module.

    """
    def __init__(self, input_dim, output_dim, num_heads=1):
        super().__init__()

        self.Wq, self.Wk, self.Wv  =\
            (nn.Linear(input_dim, output_dim * num_heads) for _ in range(3))

        self.output_dim = output_dim
        self.num_heads = num_heads
        self.Wo = nn.Linear(output_dim * num_heads, input_dim)
        self.rope_func = RoPE(output_dim)

    def forward(
            self,
            x : torch.Tensor,
            attention_mask : torch.BoolTensor = None,
            y : torch.Tensor = None,
            decoder_mask = True,
            k_cache = None,
            v_cache = None,
            start_cache = 0
        ):
        '''
            Parameters:
                x : torch.Tensor
                    The input of the Attention module
                    used at least for K, V computation
                    (for Q if decoder only)
                attention_mask : torch.BoolTensor
                    The mask for attention (True = padding token)
                y : torch.Tensor or None
                    The query input in the case of
                    cross-attention
                k_cache, v_cache : cache of this layer (B x H x max_len x N) or None
                start_cache : position of the first token of x
        '''
        if y is None:
            y = x
        B, L, _ = y.shape
        H, N = self.num_heads, self.output_dim

        # Q, K, V for all heads at once: B x L x (H*N) -> B x H x L x N
        q = self.Wq(y).view(B, L, H, N).transpose(1, 2)
        k = self.Wk(x).view(B, x.shape[1], H, N).transpose(1, 2)
        v = self.Wv(x).view(B, x.shape[1], H, N).transpose(1, 2)

        # RoPE on Q and K
        q = self.rope_func(q, start_cache)
        k = self.rope_func(k, start_cache)

        # KV cache: save the new keys/values and use all previous ones
        if k_cache is not None:
            end = start_cache + x.shape[1]
            k_cache[:, :, start_cache:end] = k
            v_cache[:, :, start_cache:end] = v
            k = k_cache[:, :, :end]
            v = v_cache[:, :, :end]
        K = k.shape[2]

        # mask: -inf on future tokens and on padding tokens
        mask = torch.zeros(B, L, K, device=x.device)
        if decoder_mask:
            mask = mask + torch.triu(torch.full((L, K), float('-inf'), device=x.device), diagonal=K - L + 1)
        if attention_mask is not None:
            mask = mask.masked_fill(attention_mask[:, None, :K], float('-inf'))

        attention_weigths, attention_output = multi_head_attention(q, k, v, mask)

        # concatenate the heads and apply Wo
        attention_output = attention_output.transpose(1, 2).reshape(B, L, H * N)
        output = self.Wo(attention_output)

        return attention_weigths, output

## Trasnsformer FeedForward

The FeedForward Network is defined using Llama 2 architecture as following for $X \in \mathbb{R^{L\times N}}$ :


1. $G = SiLU(W^gX^\intercal)$ with $W^g \in \mathbb{R}^{M \times N}$ (see SiLU in pytorch [documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html))
2. $U = W^u X^\intercal$ with $W^u  \in \mathbb{R}^{M \times N}$
3. $I = G \odot U$ (Hadamard multiplication)
4. $O = W^{o} I^{\intercal}$ with $W_{o} \in \mathbb{R}^{N \times M}$






In [6]:
class TransformerFeedForward(nn.Module):
    def __init__(self, embed_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(embed_size, intermediate_size)
        self.up_proj = nn.Linear(embed_size, intermediate_size)
        self.down_proj = nn.Linear(intermediate_size, embed_size)
        self.gate_func = nn.SiLU()

    def forward(self, x):
        gate = self.gate_func(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)

## Transformer block
We can now create the block of the

Let consider $x_0 \in \mathbb{R}^{B \times L \times N}$, the decoder block is given by :

* $x_1 = LN_1(x_0)$ (apply layer norm)
* $x_2 = Attention(x_1)$ (apply attention)
* $x_3 = x_0 + x_2$ (adding residual)
* $x_4 = LN_2(x_3)$
* $y = FF(x_4) + x_4$


In [7]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_size, intermediate_size, num_heads):
        super().__init__()
        self.attention_module =\
            RoPEAttentionModule(embed_size, embed_size, num_heads=num_heads)
        self.feed_forward = TransformerFeedForward(embed_size, intermediate_size)

        self.attention_layer_norm = nn.RMSNorm(embed_size)
        self.feed_forward_layer_norm = nn.RMSNorm(embed_size)

    def forward(
        self,
        x,
        attention_mask = None,
        k_cache=None,
        v_cache=None,
        start_cache=0,
        layer=0,
    ):
        # k_cache / v_cache contain the cache of all layers, we take the one of this layer
        layer_k = k_cache[layer] if k_cache is not None else None
        layer_v = v_cache[layer] if v_cache is not None else None

        x1 = self.attention_layer_norm(x)
        _, x2 = self.attention_module(x1, attention_mask=attention_mask,
                                      k_cache=layer_k, v_cache=layer_v,
                                      start_cache=start_cache)
        x3 = x + x2
        x4 = self.feed_forward_layer_norm(x3)
        output = self.feed_forward(x4) + x3

        return output

## Implementation notes

- **Attention:** all heads are computed in one operation instead of a loop. A mask hides future tokens and padding.
- **RoPE:** applied to Q and K to encode token positions.
- **Feed-forward:** SwiGLU, as in Llama 2.
- **Normalization:** RMSNorm before attention and before the feed-forward, as in Llama.
- **Residual:** we add the feed-forward output to x₃ (standard Llama), not to x₄ as written in the notebook.

## The Model

The model contain a stack of block on input embedding, it applies the different transformation and should return both the output embeddings and the logits (size of the vocabulary).

It should return $o_{e} \mathbb{R}^{B\times L \times N}$ and $\hat{y}\in \mathbb{R}^{B\times L \times V}$ with :

* $o_e = RMSNORM \left( b_h\circ \dots   \circ \circ b_2 \circ b_1 (w_{word\_embedding}x^\intercal) \right)$ with $b_i$ the $i^{th}$ block
* $\hat{y} = W_{lm\_head} o_e^{T}$



In [8]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocabulary_size, embed_size, intermediate_size, num_heads, hidden_layers=5, **kwargs):
        super().__init__(**kwargs)
        self.nh = num_heads
        self.nhl = hidden_layers
        self.es = embed_size

        self.wte = nn.Embedding(vocabulary_size, embed_size)

        self.blocks = nn.ModuleList([TransformerDecoderBlock(embed_size, intermediate_size, num_heads) for _ in range(hidden_layers)])
        self.ln_f = nn.LayerNorm(embed_size, eps=1e-3)
        self.lm_head = nn.Linear(embed_size, vocabulary_size)


    def forward(
            self,
            input_ids,
            attention_mask = None,
            k_cache = None,
            v_cache = None,
            start_cache = 0
        ):


        input_embed = self.wte(input_ids)
        intermediate = input_embed

        for i, block in enumerate(self.blocks):
            intermediate = block(intermediate,
                                attention_mask=attention_mask,
                                k_cache=k_cache, v_cache=v_cache,
                                start_cache=start_cache,
                                layer=i)          # so the block picks its own cache


        output_embed = self.ln_f(intermediate)
        output_lm = self.lm_head(output_embed)

        return output_embed, output_lm

##  Training the transformer

### I.Dataset
For simplicity we will use a small dataset name TinyStories, we can load it using the huggingface Datasets library as following

In [9]:
tinystories_dataset = load_dataset("roneneldan/TinyStories")
training_set = tinystories_dataset['train']
print(training_set[0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


**Dataset Exploration**

In [10]:
from collections import Counter

# Explore the TinyStories dataset: counts, labels, and class balance

dataset = tinystories_dataset

print("Dataset keys:", list(dataset.keys()))
print()

for split_name, split_ds in dataset.items():
    print(f"Split: {split_name}")
    print(f"  Number of samples: {len(split_ds)}")
    print(f"  Features: {split_ds.features}")
    print()

# Check whether the dataset has labels / target columns
all_columns = set()
for split_name, split_ds in dataset.items():
    all_columns.update(split_ds.column_names)

# Ignore text columns (usually the main input)
label_columns = [c for c in sorted(all_columns) if c != "text"]

if len(label_columns) == 0:
    print("This dataset has no explicit label columns.")
    print("It is a text-only dataset, so there is no class imbalance to measure.")
else:
    print("Potential label columns:", label_columns)
    # Count labels for each split
    for split_name, split_ds in dataset.items():
        counts = {}
        for col in label_columns:
            values = [row[col] for row in split_ds if col in row]
            if values:
                counts[col] = dict(Counter(values))
        print(f"Split {split_name} label distribution:", counts)

# Display split counts as percentages of the total dataset size
available_splits = list(dataset.keys())
split_sizes = {split_name: len(split_ds) for split_name, split_ds in dataset.items()}
total_samples = sum(split_sizes.values())
split_percentages = {split_name: (size / total_samples) * 100 for split_name, size in split_sizes.items()}

print("\nSplit sizes and percentages of total dataset:")
for split_name in available_splits:
    pct = split_percentages[split_name]
    print(f"  {split_name}: {split_sizes[split_name]} samples ({pct:.2f}%)")

Dataset keys: ['train', 'validation']

Split: train
  Number of samples: 2119719
  Features: {'text': Value('string')}

Split: validation
  Number of samples: 21990
  Features: {'text': Value('string')}

This dataset has no explicit label columns.
It is a text-only dataset, so there is no class imbalance to measure.

Split sizes and percentages of total dataset:
  train: 2119719 samples (98.97%)
  validation: 21990 samples (1.03%)


### The Dataloader

We can define the batch_size default we will use, using the DataLoader object. Here we consider that the training machine will have enough RAM for a batch of 16 samples.

In [11]:
training_dl = DataLoader(training_set, batch_size=16, shuffle=True)

### III. Tokenizer

In this exercice we will consider the LLama Tokenizer, however, you can train your own tokenizer if you prefer (see lab 2)

In [12]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model_hg_id = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_hg_id)
tokenizer.pad_token = '<pad>'
vocabulary_size = tokenizer.vocab_size

In [13]:
# we add this to test the decoder
test_model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
ids = torch.randint(0, vocabulary_size, (2, 10))
oe, oy = test_model(ids)
print(oe.shape, oy.shape)

torch.Size([2, 10, 128]) torch.Size([2, 10, 32768])


### IV. Create the model and the optimizer

In [14]:
device = 'cpu'
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
model = model.train()
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=6e-4)

In [15]:
model

TransformerDecoder(
  (wte): Embedding(32768, 128)
  (blocks): ModuleList(
    (0-7): 8 x TransformerDecoderBlock(
      (attention_module): RoPEAttentionModule(
        (Wq): Linear(in_features=128, out_features=512, bias=True)
        (Wk): Linear(in_features=128, out_features=512, bias=True)
        (Wv): Linear(in_features=128, out_features=512, bias=True)
        (Wo): Linear(in_features=512, out_features=128, bias=True)
      )
      (feed_forward): TransformerFeedForward(
        (gate_proj): Linear(in_features=128, out_features=256, bias=True)
        (up_proj): Linear(in_features=128, out_features=256, bias=True)
        (down_proj): Linear(in_features=256, out_features=128, bias=True)
        (gate_func): SiLU()
      )
      (attention_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
      (feed_forward_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
    )
  )
  (ln_f): LayerNorm((128,), eps=0.001, elementwise_affine=True)
  (lm_head): Linear(i

### VI. Training the model

In [16]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device :", device)

model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
model = model.train()
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=6e-4)

True
NVIDIA GeForce RTX 5060 Laptop GPU
Device : cuda


In [17]:
model = model.train() # Active training mode
avg_loss = []
start_time = time.time()

for i, data in enumerate(training_dl): # Path all batches
    # Tokenize the raw text
    res = tokenizer(data['text'], return_tensors="pt", padding=True, padding_side='right', max_length=128, truncation=True)
    # Offset
    x = res.input_ids[:, :-1] + 0
    y = res.input_ids[:, 1:] + 0
    # attention_mask = (res.attention_mask == 0)
    optimizer.zero_grad() # Reset gradients to zero
    oe, oy = model(x.to(device)) # Predict (return the output embedding and the vocabulary size)
    loss = loss_function(oy.to(device).view(-1, vocabulary_size), y.to(device).view(-1))
    loss.backward()
    loss_value = loss.item()
    avg_loss.append(loss_value)
    if( i%500 == 0):
        elapsed_time = time.time() - start_time
        remaining_time = int((elapsed_time/(i+1)) * (len(training_dl) - i))
        loop_sec = ((i+1)/elapsed_time)
        print(f"The loss at iteration {i+1} is {sum(avg_loss)/len(avg_loss):3.4f} remaining_time is {strftime('%H:%M:%S', gmtime(remaining_time))}s ({loop_sec:4.1f} it/s)", flush=True)
        avg_loss = []
    if( i%1000 == 0):
        torch.save(optimizer.state_dict() ,"optimizer_state_dict_llama_mini.pth.temp")
        torch.save(model.state_dict() ,"transformer_state_dict_llama_mini.pth.temp")
        shutil.copyfile("optimizer_state_dict_llama_mini.pth.temp", "optimizer_state_dict_llama_mini.pth")
        shutil.copyfile("transformer_state_dict_llama_mini.pth.temp", "transformer_state_dict_llama_mini.pth")
    optimizer.step()

The loss at iteration 1 is 10.5444 remaining_time is 22:09:12s ( 1.7 it/s)
The loss at iteration 501 is 4.1732 remaining_time is 01:46:56s (20.6 it/s)
The loss at iteration 1001 is 2.9654 remaining_time is 01:43:50s (21.1 it/s)
The loss at iteration 1501 is 2.6866 remaining_time is 01:43:12s (21.2 it/s)
The loss at iteration 2001 is 2.5436 remaining_time is 01:42:52s (21.1 it/s)
The loss at iteration 2501 is 2.4419 remaining_time is 01:42:45s (21.1 it/s)
The loss at iteration 3001 is 2.3696 remaining_time is 01:42:06s (21.1 it/s)
The loss at iteration 3501 is 2.3091 remaining_time is 01:43:40s (20.7 it/s)
The loss at iteration 4001 is 2.2640 remaining_time is 01:45:30s (20.3 it/s)
The loss at iteration 4501 is 2.2202 remaining_time is 01:47:00s (19.9 it/s)
The loss at iteration 5001 is 2.1868 remaining_time is 01:48:06s (19.7 it/s)
The loss at iteration 5501 is 2.1552 remaining_time is 01:49:20s (19.4 it/s)
The loss at iteration 6001 is 2.1241 remaining_time is 01:49:52s (19.2 it/s)
Th

## Decoding time

There is different way to decode, the simplest one is the greedy decoding method, the principle is to loop with the new token produced each time (complete `nonefficient_greedy_decoding`) and choosing the token the most likely. We can also store Keys and Values as shown in course that in practice only accellerate the generation but not change it (complete `greedy_decoding`). And finally we can also sampling new tokens (complete `sampling_decoding` ).

In [17]:
@torch.no_grad  # No backward process here, so no need to cache the generation process in memory.
def nonefficient_greedy_decoding(model, x, max_new_tokens=64):
    previous_gen = x # No need to(device), assume to be already done.
    for _ in range(max_new_tokens):
        _, logits = model(previous_gen) # return B x L x V with B the batch numbers, L the number of token in the sequence, V the vocabulary size
        # logits[:, -1, :] retrieve the scores for each vocabulary word for the last token (-1) of the sequence. We do it for all batches.
        token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # B x 1
        previous_gen = torch.cat([previous_gen, token_id], dim=-1)   # B x (L+1)
    return (previous_gen,)

@torch.no_grad
def greedy_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=None):
    previous_gen = x # No need to(device), assume to be already done.
    generated_tokens = x
    B, L = x.shape
    assert L + max_new_tokens <= max_cache_size, "Cache too small"

    # D layers, B batches, H attention heads, max_cache_size replace the L dimension (token in the sequence), N embedding size
    # For each layer, for each batch, for each attention head in that layer, for each tokens in that sequence embedded (already viewed).
    cache_shape = (model.nhl, B, model.nh, max_cache_size, model.es)
    k_cache = torch.zeros(cache_shape, device=device)
    v_cache = torch.zeros(cache_shape, device=device)

    position = 0 # at the begining we have not generated tokens yet.

    for _ in range(max_new_tokens):
        # start_cache = position of the last token.
        # With cache, no need to provide the overall updated sequence.
        _, logits = model(previous_gen, k_cache=k_cache, v_cache=v_cache, start_cache=position)
        token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # B x 1
        generated_tokens = torch.cat([generated_tokens, token_id], dim=-1)   # B x (L+1)
        previous_gen = token_id # B x L
        position = generated_tokens.shape[1]-1 # update position

        if tokenizer is not None and (token_id == tokenizer.eos_token_id).all(): # Stop the generation when we get the end of sequence token.
            break
    return (generated_tokens,)

def sampling_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=None, temperature=.7):
    """Randomly select the next token, giving a higher chance to the most probable tokens."""
    # Almost same algorithm than greedy decoding
    previous_gen = x # No need to(device), assume to be already done.
    generated_tokens = x
    B, L = x.shape
    assert L + max_new_tokens <= max_cache_size, "Cache too small"

    # D layers, B batches, H attention heads, max_cache_size replace the L dimension (token in the sequence), N embedding size
    # For each layer, for each batch, for each attention head in that layer, for each tokens in that sequence embedded (already viewed).
    cache_shape = (model.nhl, B, model.nh, max_cache_size, model.es)
    k_cache = torch.zeros(cache_shape, device=device)
    v_cache = torch.zeros(cache_shape, device=device)

    position = 0 # at the begining we have not generated tokens yet.

    for _ in range(max_new_tokens):
        # start_cache = position of the last token.
        # With cache, no need to provide the overall updated sequence.
        _, logits = model(previous_gen, k_cache=k_cache, v_cache=v_cache, start_cache=position)

        probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)   # B x V, each line sums to 1
        token_id = torch.multinomial(probs, num_samples=1)               # B x 1

        generated_tokens = torch.cat([generated_tokens, token_id], dim=-1)   # B x (L+1)
        previous_gen = token_id # B x L
        position = generated_tokens.shape[1]-1 # update position

        if tokenizer is not None and (token_id == tokenizer.eos_token_id).all(): # Stop the generation when we get the end of sequence token.
            break
    return (generated_tokens,)


**From naive greedy decoding to greedy decoding with a KV cache**

Both greedy methods generate one token at a time: at each step, the model outputs a score
(logit) for every token of the vocabulary, and the token with the highest score is appended
to the sequence (with softmax). The new sequence is then used to predict the following token, and so on.

The naive greedy decoding gives the whole sequence (prompt + tokens generated so far) to the
model at each step. Inside every attention layer, the keys K and values V of all the previous
tokens are therefore recomputed again and again, even though they were already computed at the
previous steps. The longer the text, the larger the matrices and the slower each step.

The greedy decoding with a KV cache uses a trick to avoid this computation time consumption. Thanks to
the causal mask (set to $-\infty$ all tokens not yet seen), a token never attends to the tokens that come after it, so its keys and values
never change once they have been computed. We can therefore store them in a cache
(`k_cache` and `v_cache`) the first time they are computed and reuse them at every later step.
After a first pass on the whole prompt, the model only receives the last generated token:
it computes the key and value of this single token, adds them to the cache, and computes the
attention with all the keys and values stored so far. The generated text is exactly the same as
with the naive version. Only the computation time is reduced.

**Sampling decoding**

The greedy decoding always selects the most likely token, excluding other tokens that would also
be valid and reducing the diversity of the text produced by the model. This approach is
deterministic: for the same prompt, the model will always generate exactly the same text. It also
tends to repeat itself, because the most likely continuation often leads back to the same
sentences.

The sampling decoding mitigates those limitations by drawing the next token at random
instead of always taking the most likely one. The logits are converted into probabilities with a
softmax, and a token is drawn according to these probabilities: likely tokens are chosen more
often, but less likely ones still have a chance. Each run can thus produce a different text.

A temperature parameter T sets the degree of creativity of the model. The logits are divided by T before the softmax:
- With a low temperature (T < 1), the differences between probabilities are amplified: the most likely tokens become even more likely, and the text is more coherent but less varied. When T gets close to 0, the most likely token is always chosen and we come back to greedy decoding.
- With T = 1, the tokens are drawn from the probabilities predicted by the model, without any modification.
- With a high temperature (T > 1), the probabilities become closer to each other: the creativity increases, but so does the chance of choosing unlikely tokens and producing incoherent text and hallucination.

We use T = 0.7, a common compromise between coherence and diversity.

source: https://huggingface.co/blog/mlabonne/decoding-strategies

In [18]:
import time

model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
model.load_state_dict(torch.load("transformer_state_dict_llama_mini.pth", map_location="cpu"))
model = model.eval()

decoding_method = greedy_decoding

text = "Alice"
# On utilise 'cuda' si disponible, sinon 'cpu'
run_device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(run_device)
tokenized_text = tokenizer(text, return_tensors='pt')

# On s'assure que la variable globale 'device' correspond au device d'exécution
# car la fonction greedy_decoding y fait référence directement.
device = run_device

start = time.time()
output_ids = decoding_method(model, tokenized_text.input_ids.to(run_device), max_new_tokens=64)[0]
end = time.time()
print(f'Elapsed: {end - start:.2f} seconds')
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0])

Elapsed: 0.73 seconds
Alice was a little girl who loved to play with her toys. One day, she was playing with her dolls when she heard a loud noise. She looked up and saw a big, scary monster! She was scared and ran away.

Alice was very scared and ran away. She ran and ran until she


In [19]:
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
model.load_state_dict(torch.load("transformer_state_dict_llama_mini.pth", map_location="cpu"))
model = model.eval()

decoding_method = nonefficient_greedy_decoding

text = "Alice"
run_device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(run_device)
tokenized_text = tokenizer(text, return_tensors='pt')

device = run_device
output_ids = decoding_method(model, tokenized_text.input_ids.to(run_device), max_new_tokens=64)[0]
end = time.time()
print(f'Elapsed: {end - start:.2f} seconds')
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0])

Elapsed: 2.53 seconds
Alice was a little girl who loved to play with her toys. One day, she was playing with her dolls when she heard a loud noise. She looked up and saw a big, scary monster! She was scared and ran away.

Alice was very scared and ran away. She ran and ran until she


In [20]:
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 8)
model.load_state_dict(torch.load("transformer_state_dict_llama_mini.pth", map_location="cpu"))
model = model.eval()

decoding_method = sampling_decoding

text = "Alice"
run_device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(run_device)
tokenized_text = tokenizer(text, return_tensors='pt')

device = run_device
start = time.time()
output_ids = decoding_method(model, tokenized_text.input_ids.to(run_device), max_new_tokens=64, temperature = 1.2)[0]
end = time.time()
print(f'Elapsed: {end - start:.2f} seconds')
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0])

Elapsed: 0.74 seconds
Alice and Leo were best friends who lived in the east stay safe. One sunny day, they decided to go for a walk in the park. They were both very excited. They watched when they separated these from each other to walk them down. After a while, Alice reached the end. Tonyal were the turning of


For both deterministic greedy decoding method, we observe repetition in the generated text (e.g. "She was scared and ran away."). The naive implementation take more time, we talk of seconds while in the optimized implementation we have millisecond.
The sampling decoding is fast, not so fast than the deterministic optimized method but it is the same order. The text is more diversify. At temperature 1.2, it is difficult to follow the story telling. Indeed, in the example we generated, Alice helps it father with his car, then they go to the lake for admiring dolphins (hallucination here because dolphins live in ocean), and at the end, Ted offers something to Alice.

## Evaluation ?

It remains difficult to evaluate the model. You can here propose an evalaution based on perplexity using an other model, or comparing results to ground truth using the test (here only validation) set, but it will not be totally informative !!!

In [21]:
validation_dataset = tinystories_dataset['validation']

Evaluating a text generator is difficult: a story has many valid continuations, and no single
number captures everything that makes a text "good". We therefore combine several metrics,
each measuring a different aspect of the generated text:

| Aspect | Metric |
|---|---|
| Fluency | Perplexity of our model on the validation set, perplexity given by a judge model (GPT-2) |
| Coherence with the beginning of the story | Word2vec embedding similarity, BM25 retrieval |
| Repetitions and diversity | distinct-n, type-token ratio |
| Vocabulary | Comparison of the training, validation and generated vocabularies |
| Grammar | spaCy heuristics and CYK parsing with a hand-written grammar |
| Overall quality | LLM-as-a-judge (grammar, creativity, consistency) |

**Protocol.** We take stories from the validation set, which the model has never seen. Each story
is cut in two: the first `PROMPT_TOKENS` tokens are the *prompt*, and the next `CONT_TOKENS`
tokens are the *reference continuation* (the real end of the story). The model generates a
continuation of the prompt with greedy decoding and with sampling decoding. Every metric is
computed on the three kinds of continuation (real, greedy, sampling): the real continuation is
the **baseline** that tells us what score a good text gets.

In [ ]:
#!pip install rank_bm25 gensim spacy pandas
#!python -m spacy download en_core_web_sm

   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.5 MB 4.2 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/15.5 MB 5.6 MB/s eta 0:00:03
   -------- ------------------------------- 3.1/15.5 MB 5.8 MB/s eta 0:00:03
   ----------- ---------------------------- 4.5/15.5 MB 6.0 MB/s eta 0:00:02
   -------------- ------------------------- 5.5/15.5 MB 6.0 MB/s eta 0:00:02
   ------------------ --------------------- 7.1/15.5 MB 6.1 MB/s eta 0:00:02
   --------------------- ------------------ 8.4/15.5 MB 6.1 MB/s eta 0:00:02
   ------------------------- -------------- 9.7/15.5 MB 6.2 MB/s eta 0:00:01
   ---------------------------- ----------- 11.0/15.5 MB 6.2 MB/s eta 0:00:01
   ------------------------------- -------- 12.3/15.5 MB 6.2 MB/s eta 0:00:01
   ----------------------------------- ---- 13.6/15.5 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------  15.2/15.5 MB 6.2 MB/s eta 0:00:01
  

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  ERROR: HTTP error 404 while getting https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install requirement https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz because of HTTP error 404 Client Error: Not Found for url: https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz for URL https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz


In [22]:
import spacy
from rank_bm25 import BM25Okapi
from gensim.models import Word2Vec
from transformers import AutoModelForCausalLM

nlp = spacy.load("en_core_web_sm")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()

PROMPT_TOKENS = 24   # length of the prompt given to the model
CONT_TOKENS = 80     # length of the continuation (PROMPT + CONT stays below the 128 tokens seen in training)
N_EVAL = 100         # number of validation stories used for the evaluation
N_VOCAB = 20_000     # number of stories used to build the vocabularies / train word2vec

random.seed(0)
torch.manual_seed(0)

### 1. Building the evaluation set

For each validation story long enough, we keep the prompt, the reference continuation and the two generated continuations. Only the **new** tokens are decoded for the generated texts.

In [23]:
@torch.no_grad()
def build_eval_set(n_stories=N_EVAL):
    prompts, real, greedy, sampling = [], [], [], []
    for story in validation_dataset['text']:
        if len(prompts) >= n_stories:
            break
        ids = tokenizer(story, return_tensors='pt').input_ids      # 1 x L (starts with <s>)
        if ids.shape[1] < PROMPT_TOKENS + CONT_TOKENS:
            continue                                               # story too short
        prompt_ids = ids[:, :PROMPT_TOKENS].to(device)

        out_greedy = greedy_decoding(model, prompt_ids, max_new_tokens=CONT_TOKENS, tokenizer=tokenizer)[0]
        out_sampling = sampling_decoding(model, prompt_ids, max_new_tokens=CONT_TOKENS, tokenizer=tokenizer)[0]

        prompts.append(tokenizer.decode(ids[0, :PROMPT_TOKENS], skip_special_tokens=True))
        real.append(tokenizer.decode(ids[0, PROMPT_TOKENS:PROMPT_TOKENS + CONT_TOKENS], skip_special_tokens=True))
        greedy.append(tokenizer.decode(out_greedy[0, PROMPT_TOKENS:], skip_special_tokens=True))
        sampling.append(tokenizer.decode(out_sampling[0, PROMPT_TOKENS:], skip_special_tokens=True))
    return prompts, {"real": real, "greedy": greedy, "sampling": sampling}

prompts, texts = build_eval_set()
print(len(prompts), "stories\n")
for name in texts:
    print(f"--- {name} ---\n{prompts[0]} [...] {texts[name][0]}\n")

100 stories

--- real ---
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy [...] loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.

Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy

--- greedy ---
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy [...] loved to play with his friends in the forest. They would run, jump, and laugh all day long. Roxy was very happy.

One day, Roxy saw a big tree with a hole in it. He wanted to see what was inside. Roxy thought, "I will go inside the hole and see what's inside." Roxy was very excited.

R

--- sampling ---
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy [...] was a very happy rhinoceros. He liked to play with his friends at the forest.

One day, Roxy saw a big 

In [25]:
# Words used by the text metrics: lowercase, letters and apostrophes only
# def words(text):
#     return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

def clean_text(text):
    """Minimal cleaning without removing punctuation."""
    return re.sub(r"\s+", " ", str(text)).strip()

def lexical_tokens(text):
    """retrieve lexical tokens for diversity metrics."""
    return re.findall(r"\b[\w']+\b", clean_text(text).lower(), flags=re.UNICODE)

# Training and validation subsets used for the vocabulary and for word2vec
train_texts = list(training_set.shuffle(seed=0).select(range(N_VOCAB))['text'])
val_texts = list(validation_dataset.shuffle(seed=0).select(range(min(N_VOCAB, len(validation_dataset))))['text'])

results = {}   # metric name -> {"real": ..., "greedy": ..., "sampling": ...}

### 2. Perplexity

**Perplexity of our model.** The perplexity is the exponential of the average cross-entropy loss:
$PPL = e^{loss}$. 
It measures how "surprised" the model is by a text: the lower, the better.
A perplexity of 10 means that the model is, on average, as uncertain as if it had to choose between 10 equally likely tokens. We compute it on validation stories (never seen during training) and on training stories: a much lower value on the training set would indicate overfitting.

In [26]:
@torch.no_grad()
def model_perplexity(model, dataset, n_batches=50, batch_size=32):
    model.eval()
    loss_sum = nn.CrossEntropyLoss(reduction='sum')
    total_loss, total_tokens = 0.0, 0
    for i, data in enumerate(DataLoader(dataset, batch_size=batch_size)):
        if i >= n_batches:
            break
        res = tokenizer(data['text'], return_tensors="pt", padding=True,
                        padding_side='right', max_length=128, truncation=True)
        input_ids = res.input_ids.to(device)
        attention_mask = res.attention_mask.to(device)
        x = input_ids[:, :-1]
        y = input_ids[:, 1:].clone()
        y[attention_mask[:, 1:] == 0] = -100          # padding is ignored
        _, logits = model(x)
        total_loss += loss_sum(logits.reshape(-1, vocabulary_size), y.reshape(-1)).item()
        total_tokens += (y != -100).sum().item()
    return math.exp(total_loss / total_tokens)

ppl_val = model_perplexity(model, validation_dataset)
ppl_train = model_perplexity(model, training_set.shuffle(seed=1).select(range(50 * 32)))
print(f"Perplexity on validation: {ppl_val:.2f}")
print(f"Perplexity on training  : {ppl_train:.2f}")

Perplexity on validation: 5.30
Perplexity on training  : 5.07


A perplexity of 5.3 means that, on average, the model’s uncertainty is roughly equivalent to having to choose between 5 equally probable tokens, out of a vocabulary of 32,768.

The gap between training perplexity and validation perplexity is small; therefore, the model has not overfitted.

**Perplexity given by a judge model.** Here we evaluate the *generated* texts: a larger pretrained model (GPT-2) computes the perplexity of each text (prompt + continuation). 
If the judge finds a text predictable, it looks like natural English. 
The judge has its own tokenizer, so we go back to the raw text.

*Limit:* a repetitive text ("She was happy. She was happy.") is very predictable and gets a very low perplexity although it is a bad text. This metric must be read together with distinct-n (Proportion of distinct n-grams among all generated n-grams).

In [27]:
judge_tokenizer = AutoTokenizer.from_pretrained("gpt2")
judge_lm = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

@torch.no_grad()
def judge_perplexity(text):
    enc = judge_tokenizer(text, return_tensors="pt").to(device)
    out = judge_lm(**enc, labels=enc.input_ids)       # the loss is computed by the model
    return math.exp(out.loss.item())

results["GPT-2 perplexity (median)"] = {
    name: float(np.median([judge_perplexity(p + " " + c) for p, c in zip(prompts, conts)]))
    for name, conts in texts.items()
}
results["GPT-2 perplexity (median)"]

c:\Users\jdorn\Documents\M2\NLP Today\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jdorn\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\jdorn\Documents\M2\NLP Today\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hugging

{'real': 12.504819327009708,
 'greedy': 9.24578795611028,
 'sampling': 12.28016623702651}

**What is evaluated.** For each of the 100 validation stories, we compare three continuations of the same prompt (the first 24 tokens of the story):
- **real**: the actual continuation of the story, i.e. the next 80 tokens written by the authors of the dataset. It is our **reference**: it tells us which perplexity the judge gives to natural, human-quality text from the same distribution.
- **greedy** and **sampling**: the continuations generated by our model from the same prompt, with the two decoding methods.

GPT-2 computes the perplexity of the prompt followed by each continuation. The prompt is the same for the three texts, so the differences only come from the continuations. We report the median over the 100 stories, which is less sensitive to a few outliers than the mean.

**Why we evaluate the generated texts and not the model.** The perplexity of our own model on the validation set (5.30) measures how well it *predicts* real text. It says nothing about the quality of the texts it *produces*: the same model can generate very different texts depending on the decoding method, and a model that predicts well can still produce repetitive or incoherent text
when it generates. To evaluate the generated texts, we need an external and independent model that was not trained by us: the judge. If GPT-2 finds a generated text as predictable as a real story, the generated text looks like natural English.

**Why the two perplexities cannot be compared.** Perplexity is computed per token, and the two models do not use the same tokenizer (Mistral for our model, GPT-2's own tokenizer for the judge): the same text does not have the same number of tokens for both. They were also trained on different data. A perplexity can only be compared with another perplexity computed by the same
model, which is why the generated texts are compared to the real continuations and not to the 5.30 of our model.

**Results.**

| | real | greedy | sampling |
|---|---|---|---|
| GPT-2 perplexity (median) | 12.50 | 9.25 | 12.28 |

The sampling continuations get almost the same perplexity as the real ones: according to the judge, they are as fluent as the natural English stories. The greedy continuations get a *lower* perplexity than the real ones (9.25), but this does not mean that greedy decoding writes better than the authors of the dataset. Greedy decoding always picks the most likely token, which
produces very predictable sentences and often repetitions, and a predictable text mechanically gets a low perplexity. A perplexity lower than the reference is therefore a warning sign rather than an improvement. This must be checked with the repetition metrics (distinct-n): a low perplexity combined with a low distinct-n means that the text is predictable because it repeats itself, not because it is well written.

### 3. Coherence with the prompt: embedding similarity

We train a small word2vec model on training stories, so that the vectors are adapted to the vocabulary of TinyStories. A text is represented by the average of the vectors of its words, and we compute the cosine similarity between the prompt and its continuation. Unlike a lexical metric, word2vec knows that "dog" and "puppy" are close.

To interpret the values, we add a **random baseline**: the similarity between a prompt and the real continuation of *another* story. A model that stays on topic should be well above it.
We also compute the similarity between the generated continuation and the real one.

In [30]:
w2v = Word2Vec(sentences=[lexical_tokens(t) for t in train_texts], vector_size=100,
               window=5, min_count=3, workers=4, epochs=5, seed=0)

def text_vector(text):
    vecs = [w2v.wv[w] for w in lexical_tokens(text) if w in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na > 0 and nb > 0 else 0.0

prompt_vecs = [text_vector(p) for p in prompts]
cont_vecs = {name: [text_vector(c) for c in conts] for name, conts in texts.items()}

results["Similarity prompt / continuation"] = {
    name: float(np.mean([cosine(p, c) for p, c in zip(prompt_vecs, vecs)]))
    for name, vecs in cont_vecs.items()
}
results["Similarity generated / real continuation"] = {
    name: float(np.mean([cosine(r, c) for r, c in zip(cont_vecs["real"], vecs)]))
    for name, vecs in cont_vecs.items()
}
shifted = cont_vecs["real"][1:] + cont_vecs["real"][:1]       # continuation of another story
random_baseline = float(np.mean([cosine(p, c) for p, c in zip(prompt_vecs, shifted)]))

print("Random baseline (prompt / continuation of another story):", round(random_baseline, 3))
random_baseline_cont = {
    name: float(np.mean([cosine(r, c) for r, c in zip(cont_vecs["real"], vecs[1:] + vecs[:1])]))
    for name, vecs in cont_vecs.items()
}
print("Random baseline (continuation / real continuation of another story):", random_baseline_cont)

pd.DataFrame(results).T.loc[["Similarity prompt / continuation", "Similarity generated / real continuation"]]


Random baseline (prompt / continuation of another story): 0.173
Random baseline (continuation / real continuation of another story): {'real': 0.5600410759449005, 'greedy': 0.5376963997818529, 'sampling': 0.5582382974028587}


,real,greedy,sampling
Similarity prompt / continuation,0.255924,0.269312,0.264164
Similarity generated / real continuation,1.000000,0.681031,0.662404


**Random baseline for Prompt / continuation.** All the TinyStories texts are written with the same small vocabulary and the same style, so two stories that have nothing to do with each other already have a non-zero similarity. A similarity score can therefore not be interpreted on its own: we need to know which value two unrelated texts get anyway.

The random baseline measures this baseline level. Each prompt is paired with the continuation of a different story, that is, with a text that does not address the same subject. More precisely, the list of continuations is shifted by one position: the prompt of story 0 is paired with the continuation of story 1, the prompt of story 1 with the continuation of story 2, and so on, until
the last prompt, which is paired with the continuation of story 0. The similarity of each of these "mismatched" pairs is then calculated, and the scores are averaged. With our 100 stories, this gives us an average similarity of 0.173. 

Therefore, a similarity computed between a prompt and its continuation only indicates a real link if it is clearly above this value. This is the case for the real continuations (0.249) and for the generated ones (0.266 for greedy, 0.259 for sampling): the continuations do talk about the subject of their prompt (same characters, same objects, and places as the prompt). The slightly higher value of greedy decoding is probably due to its tendency to repeat words of the
prompt, such as the name of the character, rather than to a better coherence.


**Generated / real continuation.**
For the second random baseline, we apply the same principle to the comparison between continuations: each continuation (real, greedy or sampling) is paired with the real continuation of the next story. 

Two real continuations of different stories already show a similarity of 0.560
(0.538 for greedy, 0.558 for sampling). This baseline is much higher than the first one because continuations are longer than prompts: the longer the texts, the closer their averaged word vectors get to the "average story", and the more two random texts look alike. 
If we had interpreted the 0.677 of greedy decoding in isolation, we might have assumed a very strong connection with the real continuation, whereas most of this score simply stems from the fact that all the stories resemble
one another. What matters is the gap with the baseline: a generated continuation is closer to the real continuation of its own story than to the continuation of another story (+0.14 for greedy, +0.10 for sampling). The model therefore goes in the same direction as the original story, but this link is more modest than the raw values suggest.
the raw value.

**Limit.** These similarities measure whether two texts talk about the same things, not whether the story is logical: a continuation can reuse all the right characters and objects and still tell something absurd. Logical coherence is evaluated later and can be evaluated with an LLM as judge.


### 4. Coherence with the prompt: BM25 retrieval

BM25 is a ranking function from information retrieval: it scores a document against a query according to the words they share, weighted by their frequency and their rarity in the corpus.
We use it as a **retrieval task**: all the continuations form the corpus, each prompt is a query, and we look at the rank of *its own* continuation among all of them. If the continuation reuses the characters, objects and places of the prompt, it should be ranked first.

We report the top-1 accuracy (own continuation ranked first) and the Mean Reciprocal Rank (MRR, average of 1 / rank; 1 is perfect, about 0.05 is random with 100 stories). Stop words are removed, since they carry no information about the topic. 

*Limit:* BM25 is purely lexical, "dog" and "puppy" do not match.

In [36]:
STOP_WORDS = nlp.Defaults.stop_words

def content_words(text):
    return [w for w in lexical_tokens(text) if w not in STOP_WORDS]

def bm25_retrieval(prompts, continuations):
    bm25 = BM25Okapi([content_words(c) or ["<empty>"] for c in continuations])
    ranks = []
    for i, prompt in enumerate(prompts):
        scores = bm25.get_scores(content_words(prompt))
        # rank of the right continuation (ties count against it: identical texts cannot be told apart)
        ranks.append(int((scores >= scores[i]).sum()))
    ranks = np.array(ranks)
    return float((ranks == 1).mean()), float((1 / ranks).mean())

bm25_results = {name: bm25_retrieval(prompts, conts) for name, conts in texts.items()}
results["BM25 top-1 accuracy"] = {name: r[0] for name, r in bm25_results.items()}
results["BM25 MRR"] = {name: r[1] for name, r in bm25_results.items()}
pd.DataFrame(results).T.loc[["BM25 top-1 accuracy", "BM25 MRR"]]

,real,greedy,sampling
BM25 top-1 accuracy,0.480000,0.440000,0.440000
BM25 MRR,0.579937,0.545968,0.537544


| | real | greedy | sampling |
|---|---|---|---|
| BM25 top-1 accuracy | 0.48 | 0.44 | 0.44 |
| BM25 MRR | 0.580 | 0.546 | 0.538 |

In this retrieval task, each query (a prompt) has exactly one relevant document: its own continuation. With a single relevant document, the usual ranking metrics reduce to the ones we use:
recall@1 is the top-1 accuracy (measures wether the information is ranked first), and NDCG ($1/\log_2(rank+1)$) follows the same idea as the MRR ($1/rank$) with a different penality depending on the rank. The MRR is the standard metric when there is a single correct answer.

The retrieval has its own built-in baseline: picking a continuation at random among the 100 would give a top-1 accuracy of 1% and an MRR of about 0.05. The real continuations are retrieved first in 48% of the cases: the task is far from trivial, because the same first names and objects appear in
many stories and a continuation often introduces new elements absent from the 24 tokens of the prompt. The generated continuations are almost as well retrieved (44%, i.e. 4 stories out of 100 fewer than the real ones). BM25 thus confirms the conclusion of the embedding similarity, with a lexical point of view: the generated continuations reuse the exact words of their prompt (names of characters, objects, places), and are almost as specific to their prompt as the real continuations.

### 5. Repetitions and diversity

**distinct-n** is the proportion of unique n-grams (groups of n consecutive words) in a text: 1 means no repetition, a value close to 0 means that the text loops. 
We compute it for each text (repetitions *inside* a story) and on all the texts together (*corpus-level*: do the stories all look alike?).

The **type-token ratio** (TTR) is the number of different words divided by the number of words.
It decreases naturally with the length of the text, so we compute it on the first
`TTR_WORDS` words of each text only, to compare texts of the same length.

In [37]:
def ngrams(word_list, n):
    return [tuple(word_list[i:i + n]) for i in range(len(word_list) - n + 1)]

def distinct_n(text, n):
    grams = ngrams(lexical_tokens(text), n)
    return len(set(grams)) / len(grams) if grams else 0.0

def corpus_distinct_n(text_list, n):
    grams = [g for t in text_list for g in ngrams(lexical_tokens(t), n)]
    return len(set(grams)) / len(grams) if grams else 0.0

TTR_WORDS = 40
def ttr(text, n_words=TTR_WORDS):
    w = lexical_tokens(text)[:n_words]
    return len(set(w)) / len(w) if len(w) == n_words else None   # None: text too short

for n in (1, 2, 3):
    results[f"distinct-{n}"] = {name: float(np.mean([distinct_n(t, n) for t in conts]))
                               for name, conts in texts.items()}
results["corpus distinct-2"] = {name: corpus_distinct_n(conts, 2) for name, conts in texts.items()}
results[f"TTR ({TTR_WORDS} words)"] = {
    name: float(np.mean([v for v in map(ttr, conts) if v is not None] or [np.nan]))
    for name, conts in texts.items()
}
pd.DataFrame(results).T.loc[["distinct-1", "distinct-2", "distinct-3", "corpus distinct-2", f"TTR ({TTR_WORDS} words)"]]

,real,greedy,sampling
distinct-1,0.687802,0.639158,0.677069
distinct-2,0.948369,0.901838,0.934625
distinct-3,0.989435,0.960958,0.983449
corpus distinct-2,0.630104,0.414819,0.529892
TTR (40 words),0.765500,0.728000,0.758250


| | real | greedy | sampling |
|---|---|---|---|
| distinct-1 | 0.688 | 0.639 | 0.677 |
| distinct-2 | 0.948 | 0.902 | 0.935 |
| distinct-3 | 0.989 | 0.961 | 0.983 |
| corpus distinct-2 | 0.630 | 0.415 | 0.530 |
| TTR (40 words) | 0.766 | 0.728 | 0.758 |

**Inside a story**, the generated texts contain few repetitions, close to the real ones. distinct-1 is naturally lower than 1 for all texts, including the real ones, since isolated words such as "the", "she", "was" or the name of the character are necessarily repeated in a story. For bigrams and trigrams, the values of the generated texts are close to the real ones (0.90 and 0.93 against 0.95 for distinct-2), with slightly more repetitions for greedy decoding. 
The TTR measures the lexical richness of the first 40 words: about 31 different words out of 40 for the real texts, and similar values for the generated ones.

**Across stories**, the corpus-level distinct-2 shows the main weakness of greedy decoding: 0.41 against 0.63 for the real continuations. Greedy decoding reuses the same word pairs from one story to another: since it is deterministic, similar contexts always lead to the same choices, which
produces stock phrases shared by many stories. Its stories therefore look alike, although each story taken separately contains few repetitions. Sampling reduces this effect (0.53) without reaching the diversity of the real stories. This also explains part of the low judge perplexity of
greedy decoding: stock phrases are very predictable.

### 6. Vocabulary: training vs validation vs generated

TinyStories was written with a deliberately small vocabulary (the words a 3-4 year old child understands), so comparing to the 32,768 tokens of the Mistral tokenizer would not be meaningful.
We compare instead the vocabularies of the texts themselves:

* **training vs validation** (same number of stories): if the validation set uses almost only words seen in training, the model has no reason to fail because of unknown words.
* **generated vs real continuations** (same number of texts): does the model use a vocabulary as rich as the real stories? The *OOV rate* (out of vocabulary) is the proportion of words that never appear in the training stories: for generated texts, these are mostly invented words, produced when the model glues subword tokens incorrectly.

In [39]:
def vocab_counter(text_list):
    return Counter(w for t in text_list for w in lexical_tokens(t))

def oov_rate(counter, reference_vocab):
    total = sum(counter.values())
    return sum(c for w, c in counter.items() if w not in reference_vocab) / total if total else 0.0

train_vocab = vocab_counter(train_texts)
val_vocab = vocab_counter(val_texts)

print(f"Training   ({len(train_texts)} stories): {len(train_vocab):6d} distinct words")
print(f"Validation ({len(val_texts)} stories): {len(val_vocab):6d} distinct words")
print(f"Validation words also in training vocabulary: {len(set(val_vocab) & set(train_vocab)) / len(val_vocab):.1%}")
print(f"OOV rate of validation (word occurrences)   : {oov_rate(val_vocab, train_vocab):.2%}")
print("Most frequent validation words unseen in training:",
      [w for w, _ in val_vocab.most_common() if w not in train_vocab][:15])

results["distinct words used"] = {name: len(vocab_counter(conts)) for name, conts in texts.items()}
results["OOV rate vs training"] = {name: oov_rate(vocab_counter(conts), train_vocab)
                                   for name, conts in texts.items()}
pd.DataFrame(results).T.loc[["distinct words used", "OOV rate vs training"]]

Training   (20000 stories):  12377 distinct words
Validation (20000 stories):  12160 distinct words
Validation words also in training vocabulary: 77.2%
OOV rate of validation (word occurrences)   : 0.17%
Most frequent validation words unseen in training: ['eva', 'foxy', 'louise', 'annabel', 'jennie', 'carly', 'dotty', 'woodpecker', 'zara', 'bison', 'henny', 'tucker', 'izzy', 'catie', 'ameli']


,real,greedy,sampling
distinct words used,965.000000,576.000000,753.00000
OOV rate vs training,0.001512,0.001211,0.00205


In [40]:
# Words invented by the model (never seen in the training stories)
for name in ("greedy", "sampling"):
    invented = [w for w, _ in vocab_counter(texts[name]).most_common() if w not in train_vocab]
    print(f"{name:9s}: {invented[:20]}")

greedy   : ['ummy', 'don', 'beecuts', "toot's", 'ens', 'ts']
sampling : ['ummy', 'raky', "buzzy's", 'lull', 'ens', 'ts', 'sharper', 'cush', 'th', 'oliver', 'michelar']


In [44]:
for p, c in zip(prompts, texts["greedy"]):
    if lexical_tokens(c)[:1] and lexical_tokens(c)[0] in ("ummy", "ens", "ts", "th"):
        print(repr(p[-30:]), "|", repr(c[:30]))


'o play with her toys and eat y' | "ummy snacks. One day, Lily's m"
'mmy had a black cat named Mitt' | 'ens. One day, Timmy and Mitten'
'enny. Benny loved to eat carro' | 'ts and play with his friends. '
' eat sugar because it was so y' | "ummy. One day, Lily's mom made"


**Training vs validation.** The two vocabularies have similar sizes (12,377 and 12,160 distinct words for 20,000 stories each). Only 77% of the distinct validation words appear in the training vocabulary, but they represent 99.8% of the word occurrences (OOV rate of 0.17%): the unseen words are very rare words, mostly first names (Eva, Louise, Zara...) and a few uncommon animals (bison, woodpecker), appearing once or twice.
In practice, almost all the words of a validation story were seen during training.

**Generated vs real continuations.** The real continuations use 965 distinct words, against 753 for sampling and 576 for greedy decoding. Greedy decoding is limited to the most likely words, which confirms the lack of diversity across stories observed with the corpus-level distinct-2. Sampling broadens the vocabulary without reaching the richness of the real texts: with a temperature of 0.7, the probabilities are sharpened towards frequent words, and a higher temperature would broaden the vocabulary further at the cost of coherence.

**Unseen words in the generated texts.** The OOV rates are very low and similar to the real continuations (0.12% for greedy, 0.21% for sampling, 0.15% for the real ones). Unlike the validation set, the unseen words of the generated texts are mostly not first names. They fall into three categories: fragments of words such as "ummy", "ens" or "th", which most likely come from our protocol (the prompt is cut at exactly 24 tokens, sometimes in the middle of a word, and only the continuation is decoded: the model completes "Y" into "Yummy", which gives "ummy". The same fragments appear for both decoding methods, which start from the same prompts). There are a few invented words produced by badly assembled subword tokens ("beecuts", "raky", "cush") and correct words absent from the 20,000 training stories ("lull", "sharper"). The rate of truly invented words is therefore even lower than the OOV rate suggests.

### 7. Grammar

spaCy alone cannot tell whether a sentence is correct: its tagger was trained on correct text and always tries to find an interpretation (in "He went the.", it tags *the* as a pronoun). We therefore combine two complementary approaches.

**CYK parsing.** spaCy only gives the part-of-speech tag of each word (DET, NOUN, VERB...), and the CYK algorithm checks whether this sequence of tags can be generated by a context-free grammar written in Chomsky normal form. If no tree with the axiom `S` covers the whole sentence,  the sentence is considered ungrammatical. Two adaptations of the tags:
* the articles *the*, *a*, *an* are always tagged `DET` (spaCy sometimes "repairs" broken sentences by changing their tag);
* tokens can receive extra tags (the CYK input is a list of tags per word): `PRT` for verb particles (*picked **up***) and `PARTICIPLE` for participles (*a girl **named** Lily*).

**spaCy heuristics.** A grammar on part-of-speech tags cannot see some errors, so we add simple checks based on spaCy's analysis: the same word repeated twice in a row, a sentence without verb, and subject-verb agreement in the present tense (*she play*, *they plays*).

**Protocol.** The continuations start and end in the middle of a sentence (they are cut at a fixed number of tokens), so the first and last sentences of each text are removed. Sentences with dialogue (quotation marks) and sentences longer than 25 words are skipped: our small grammar does not cover them, and the number of parse trees grows quickly with the length of the sentence.

**The grammar only covers a small part of English.** A sentence rejected by CYK can be correct but use a construction absent from the grammar. That is why the **parse rate of the real continuations is the reference**: what matters is the gap between generated and real texts, not the absolute value.

In [31]:
class Node():
    def __init__(self, type, left, right = None, word=None):
        self.type = type
        self.left = left
        self.right = right
        self.word = word

    def _recursive_repr(self, tab=0):
        s =  f"{'----' * tab}{self.type}"
        if self.word is not None:
            s += f" - {self.word}"
        s+= "\n"
        if self.left is not None:
            s+=  f"{self.left._recursive_repr(tab+1)}"
        if self.right is not None:
            s+=  f"{self.right._recursive_repr(tab+1)}"
        return s

    def __str__(self):
        return self._recursive_repr() + "\n"

    def __repr__(self,):
        return self._recursive_repr() + "\n"

class CYK():
    def __init__(self, grammar):
        """ A simple implementation of CYK algorithm (grammar in Chomsky normal form) """
        self.grammar = grammar

    def __call__(self, input_list, str):
        size = len(input_list)
        cyk_tab = [ [[] for _ in range(i+1)] for i in range(len(input_list)) ][::-1]

        # process terminals
        for i, tags_list in enumerate(input_list):
            for tag in tags_list:
                for production, rule in self.grammar:
                    if(len(rule) == 1) and (tag in rule):
                        cyk_tab[0][i].append(Node(production, None, word=str[i]))
        for cyk_depth in range(2, size+1):
            left_levels = list(range(1, cyk_depth))
            right_levels = list(range(1, cyk_depth)[::-1])
            for start in range(0, size - cyk_depth + 1):
                for left_level, right_level in zip(left_levels, right_levels):
                    left_start = start
                    right_start = start + left_level
                    Al, Bl = cyk_tab[left_level-1][left_start], cyk_tab[right_level-1][right_start]
                    for p, j in self.grammar:
                        if(len(j) == 2):
                            l, r = j
                            A = [a for a in Al if(a.type == l)]
                            B = [b for b in Bl if(b.type == r)]
                            cyk_tab[cyk_depth-1][start] += [Node(p, a, b ) for a in A  for b in B]

        return [i for i in cyk_tab[-1][0]]

In [32]:
# Context-free grammar in Chomsky normal form, on spaCy universal POS tags
GRAMMAR = [
    # ---------- sentence level ----------
    ("S", ("NP", "VP")),          # The cat | sat on the couch
    ("S", ("S", "CS")),           # S | and she smiled / because she was sad
    ("CS", ("CCONJ", "S")),
    ("CS", ("SCONJ", "S")),
    ("S", ("SUB", "S")),          # When it rained | she stayed home
    ("SUB", ("SCONJ", "S")),
    ("S", ("ADV", "S")),          # Then | she ran
    ("S", ("PP", "S")),           # Once upon a time | there was a girl
    ("S", ("TIME", "S")),         # One day | Lily went out
    ("TIME", ("NUM", "NOUN")),
    ("TIME", ("DET", "NOUN")),    # Every day / That night
    # ---------- noun phrases ----------
    ("NP", ("DET", "NOM")),       # the | big dog
    ("NP", ("PRON", "NOM")),      # her | ball (possessive)
    ("NP", ("NUM", "NOM")),       # two | cats
    ("NP", ("NP", "PP")),         # a girl | with a hat
    ("NP", ("NP", "CNP")),        # Tom | and Lily
    ("CNP", ("CCONJ", "NP")),
    ("NP", ("NP", "VPART")),      # a girl | named Lily
    ("VPART", ("VPART", "NP")),
    ("VPART", ("VPART", "PP")),
    ("NOM", ("ADJ", "NOM")),      # big | red ball
    ("NOM", ("NOUN", "NOM")),     # toy | car
    # ---------- verb phrases ----------
    ("VP", ("VP", "NP")),         # saw | a bird
    ("VP", ("VP", "PP")),         # sat | on the couch
    ("VP", ("VP", "ADV")),        # ran | fast
    ("VP", ("VP", "PRT")),        # picked | up
    ("VP", ("ADV", "VP")),        # quickly | ran
    ("VP", ("AUX", "VP")),        # was | playing
    ("VP", ("AUX", "ADJP")),      # was | happy
    ("VP", ("AUX", "NP")),        # was | a girl
    ("VP", ("AUX", "PP")),        # was | in the park
    ("VP", ("PART", "VP")),       # not | go
    ("VP", ("VP", "INF")),        # wanted | to play
    ("INF", ("PART", "VP")),
    ("VP", ("VP", "ADJP")),       # felt | happy
    ("VP", ("VP", "CVP")),        # ran | and jumped
    ("CVP", ("CCONJ", "VP")),
    ("VP", ("VP", "CADV")),       # up | and down
    ("CADV", ("CCONJ", "ADV")),
    ("VP", ("VP", "S")),          # said | it was fun
    ("VP", ("VP", "CS")),         # jumped | because he was happy
    # ---------- adjectives / prepositions ----------
    ("ADJP", ("ADV", "ADJP")),    # very | happy
    ("ADJP", ("ADJP", "INF")),    # happy | to help
    ("ADJP", ("ADJP", "CS")),     # so excited | that he jumped
    ("PP", ("ADP", "NP")),        # on | the couch
    ("PP", ("ADV", "PP")),        # once | upon a time
    # ---------- terminals: tag -> category ----------
    ("DET", ("DET",)), ("NOUN", ("NOUN",)), ("NOM", ("NOUN",)),
    ("NP", ("NOUN",)), ("NP", ("PRON",)), ("NP", ("PROPN",)),
    ("PRON", ("PRON",)), ("ADJ", ("ADJ",)), ("ADJP", ("ADJ",)), ("NOM", ("ADJ",)),
    ("VP", ("VERB",)), ("AUX", ("AUX",)), ("VP", ("AUX",)),
    ("ADV", ("ADV",)), ("ADP", ("ADP",)), ("ADP", ("SCONJ",)), ("PART", ("PART",)),
    ("CCONJ", ("CCONJ",)), ("SCONJ", ("SCONJ",)), ("NUM", ("NUM",)),
    ("PRT", ("PRT",)), ("VPART", ("PARTICIPLE",)),
]
cyk = CYK(GRAMMAR)

SKIPPED_TAGS = {"PUNCT", "SPACE", "SYM", "INTJ", "X"}
FORCED_TAGS = {"the": "DET", "a": "DET", "an": "DET"}
MAX_SENTENCE_WORDS = 25

def cyk_tags(token):
    tags = [FORCED_TAGS.get(token.lower_, token.pos_)]
    if token.dep_ == "prt":
        tags.append("PRT")
    if token.pos_ == "VERB" and token.tag_ in ("VBN", "VBG"):
        tags.append("PARTICIPLE")
    return tags

def cyk_parse(sentence):
    tokens = [t for t in sentence if t.pos_ not in SKIPPED_TAGS]
    trees = cyk([cyk_tags(t) for t in tokens], [t.text for t in tokens])
    return [tree for tree in trees if tree.type == "S"]

# Quick check on hand-written sentences
for s in ["Once upon a time, there was a little girl named Lily.",
          "She loved to play with her big red ball in the park.",
          "The the dog ran.", "He to the park went the."]:
    print("OK " if cyk_parse(nlp(s)) else "-- ", s)
print()
print(cyk_parse(nlp("The cat sat on the couch."))[0])

OK  Once upon a time, there was a little girl named Lily.
OK  She loved to play with her big red ball in the park.
--  The the dog ran.
--  He to the park went the.

S
----NP
--------DET - The
--------NOM - cat
----VP
--------VP - sat
--------PP
------------ADP - on
------------NP
----------------DET - the
----------------NOM - couch




In [33]:
SING_3RD = {"he", "she", "it"}
NOT_SING_3RD = {"i", "you", "we", "they"}

def spacy_errors(sentence):
    errors = []
    tokens = [t for t in sentence if not t.is_punct and not t.is_space]
    # 1. the same word twice in a row ("the the", "bird bird")
    for a, b in zip(tokens, tokens[1:]):
        if a.lower_ == b.lower_ and a.lower_ not in {"very", "so", "ha"}:
            errors.append(f"repeated word '{a.text}'")
    # 2. sentence without any verb
    if not any(t.pos_ in ("VERB", "AUX") for t in tokens):
        errors.append("no verb")
    # 3. subject-verb agreement in the present tense
    for t in tokens:
        if t.dep_ == "nsubj" and t.head.tag_ in ("VBP", "VBZ"):
            singular = t.lower_ in SING_3RD or t.tag_ in ("NN", "NNP")
            plural = t.lower_ in NOT_SING_3RD or t.tag_ in ("NNS", "NNPS")
            if (t.head.tag_ == "VBP" and singular) or (t.head.tag_ == "VBZ" and plural):
                errors.append(f"agreement '{t.text} ... {t.head.text}'")
    return errors

for s in ["She play with the ball.", "They plays in the park.", "Lily saw a bird bird.", "The big red ball."]:
    print(s, spacy_errors(nlp(s)))

She play with the ball. ["agreement 'She ... play'"]
They plays in the park. ["agreement 'They ... plays'"]
Lily saw a bird bird. ["repeated word 'bird'"]
The big red ball. ['no verb']


In [34]:
def grammar_evaluation(text_list):
    n_sentences, n_evaluated, n_parsed, n_errors = 0, 0, 0, 0
    error_examples, rejected_examples = [], []
    for doc in nlp.pipe(text_list):
        sentences = list(doc.sents)[1:-1]          # first and last sentences are cut
        for sent in sentences:
            n_sentences += 1
            errors = spacy_errors(sent)
            n_errors += len(errors) > 0
            if errors and len(error_examples) < 5:
                error_examples.append((sent.text, errors))

            n_words = sum(1 for t in sent if t.pos_ not in SKIPPED_TAGS)
            if '"' in sent.text or n_words < 2 or n_words > MAX_SENTENCE_WORDS:
                continue                            # dialogue or length not covered by the grammar
            n_evaluated += 1
            if cyk_parse(sent):
                n_parsed += 1
            elif len(rejected_examples) < 5:
                rejected_examples.append(sent.text)
    return {
        "sentences": n_sentences,
        "spaCy error rate": n_errors / max(n_sentences, 1),
        "CYK evaluated sentences": n_evaluated,
        "CYK parse rate": n_parsed / max(n_evaluated, 1),
        "error examples": error_examples,
        "rejected examples": rejected_examples,
    }

grammar = {name: grammar_evaluation(conts) for name, conts in texts.items()}
results["spaCy error rate (sentences)"] = {name: g["spaCy error rate"] for name, g in grammar.items()}
results["CYK parse rate"] = {name: g["CYK parse rate"] for name, g in grammar.items()}

for name, g in grammar.items():
    print(f"=== {name}: {g['sentences']} sentences, {g['CYK evaluated sentences']} evaluated by CYK")
    print("  spaCy errors:", *g["error examples"], sep="\n    ")
    print("  rejected by CYK:", *g["rejected examples"], sep="\n    ")
pd.DataFrame(results).T.loc[["spaCy error rate (sentences)", "CYK parse rate"]]

=== real: 487 sentences, 401 evaluated by CYK
  spaCy errors:
    ('He wanted to play with it, but when he touched it, it started to roll away.', ["repeated word 'it'"])
    ('The ball went far, far away.', ["repeated word 'far'"])
    ('So, he swam and swam until he found a big rock.', ["agreement 'he ... swam'"])
    ('"Hi, Finny!', ['no verb'])
    ('"Wow, Lily!', ['no verb'])
  rejected by CYK:
    It was shiny and cold, and she wanted to climb it.


    Roxy tried to climb the icy hill, but it was very slippery.
    She tried again and again, but she kept falling down.
    The candle was her mom's favorite.
    One sunny day, they went to the park to play.


=== greedy: 531 sentences, 392 evaluated by CYK
  spaCy errors:
    ('So, he swam up to the fish and said, "Hi, fish!', ["agreement 'he ... swam'"])
    ('Billy asked Benny if he could borrow his carrot carrot.', ["repeated word 'carrot'"])
    ('"Hi Max!', ['no verb'])
    ('A big hole!"', ['no verb'])
  rejected by CYK:
    

,real,greedy,sampling
spaCy error rate (sentences),0.010267,0.007533,0.014815
CYK parse rate,0.713217,0.798469,0.780488


**Interpretation of the grammar metrics.**

| | real | greedy | sampling |
|---|---|---|---|
| spaCy error rate (sentences) | 1.0% | 0.8% | 1.5% |
| CYK parse rate | 71.3% | 79.8% | 78.0% |

The spaCy error rate is about 1% for the three kinds of text, including the real continuations.
Since the real stories are correctly written, this 1% mostly corresponds to false alarms of our simple rules (for instance an exclamation such as "Oh no!" has no verb), and it gives the noise level of the metric. 
With a few hundred sentences per column, 1% only represents a handful of
sentences, so the differences between the three columns are too small to be interpreted. The difference between 0.8% and 1.5% can come from one or two sentences, which is not representative. The slightly higher rate of sampling (1.5%) is consistent with the fact that it sometimes picks less likely tokens, but it is not significant.

To illustrate this, here is some spacy errors:
- ('He wanted to play with it, but when he touched it, it started to roll away.', ["repeated word 'it'"])
- ('The ball went far, far away.', ["repeated word 'far'"])
- ('So, he swam and swam until he found a big rock.', ["agreement 'he ... swam'"])
- ('"Hi, Finny!', ['no verb'])
- ('"Wow, Lily!', ['no verb'])

Most of those errors are correct sentences misclassified by SpaCy because of the grammar limitations.

The CYK parse rate of the real continuations is 71%: as expected, our small grammar does not cover all the English constructions, and the rejected real sentences are correct sentences outside its scope. 
The generated continuations get a higher parse rate (80% for greedy, 78% for sampling). This does not mean that they are more grammatical than the real stories, but rather that they are syntactically simpler: the model tends to reproduce the most frequent constructions of TinyStories, which are precisely the ones covered by the grammar, while the real stories use more varied structures. 
This effect is stronger for greedy decoding, which always picks the most likely token and repeats simple sentences. As for the judge perplexity, a score better than the reference is a sign of predictability rather than of quality.

Overall, these two metrics do not detect any grammatical degradation in the generated texts. They are however too coarse to detect small differences, and are completed by the grammar grade of the LLM judge.

### 8. LLM-as-a-judge

We ask an instruction-tuned model to grade each
continuation on three criteria from 1 to 10: grammar, creativity and consistency with the
beginning of the story. The judge sees the beginning of the story and the continuation, separated
by `***`, and must answer in JSON.

We use a small open model so that the evaluation runs locally (on a subset of stories, since it is slow). 

*Limits:* a small judge is noisy and sometimes inconsistent; a larger model (or an API) would give more reliable grades. Grading the real continuations tells us how generous the judge
is.

In [42]:
JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # can be replaced by a larger instruct model
N_JUDGE = 30

import json

llm_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
llm_judge = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype="auto").to(device).eval()

JUDGE_PROMPT = '''The following is the beginning of a short story written for young children, followed by a continuation written by a student. The symbol *** separates the beginning from the continuation. The continuation may stop in the middle of a sentence; do not penalize this.

{prompt} *** {continuation}

Grade the continuation from 1 (very bad) to 10 (excellent) on three criteria:
- grammar: are the sentences grammatically correct?
- creativity: is the continuation original and interesting?
- consistency: is it consistent with the beginning of the story (characters, events, logic)?

Answer only with a JSON object, for example: {{"grammar": 5, "creativity": 5, "consistency": 5}}'''

@torch.no_grad()
def llm_grade(prompt, continuation):
    messages = [{"role": "user", "content": JUDGE_PROMPT.format(prompt=prompt, continuation=continuation)}]
    inputs = llm_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(device)
    out = llm_judge.generate(**inputs, max_new_tokens=60, do_sample=False)
    answer = llm_tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\{.*?\}", answer, re.DOTALL)
    try:
        grades = json.loads(match.group(0))
        return {k: float(grades[k]) for k in ("grammar", "creativity", "consistency")}
    except (AttributeError, KeyError, ValueError, TypeError):
        return None                                  # the judge did not answer in the right format

judge_grades = {}
for name, conts in texts.items():
    grades = [llm_grade(p, c) for p, c in zip(prompts[:N_JUDGE], conts[:N_JUDGE])]
    valid = [g for g in grades if g is not None]
    print(f"{name}: {len(valid)}/{len(grades)} valid answers")
    judge_grades[name] = {k: float(np.mean([g[k] for g in valid])) for k in ("grammar", "creativity", "consistency")}

for criterion in ("grammar", "creativity", "consistency"):
    results[f"LLM judge: {criterion} (/10)"] = {name: g[criterion] for name, g in judge_grades.items()}
pd.DataFrame(judge_grades)

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 113.76it/s]


real: 30/30 valid answers
greedy: 30/30 valid answers
sampling: 30/30 valid answers


,real,greedy,sampling
grammar,6.833333,7.0,7.0
creativity,6.233333,6.4,6.4
consistency,7.733333,8.1,8.0


The judge answered in the expected format for all the 30 stories of each column. The grades of the generated continuations are equal to or slightly higher than those of the real continuations on the three criteria. Rather than generated texts better than the real stories, this most likely shows the
limits of the judge itself. The differences between columns do not exceed 0.4 points on 30 stories, which is probably within the noise of the grades, and all the grades lie between 6 and 8. The real continuations, which are correctly written, only get 6.8 in grammar, which also shows that the judge is not fully reliable.

Two observations are consistent with the other metrics. First, the judge reads each story separately, so it cannot see that greedy decoding reuses the same expressions from one story to another, the main weakness revealed by the corpus-level distinct-2: this is why greedy decoding gets a creativity grade as high as sampling. Second, the generated continuations use simpler sentences (higher CYK parse rate) and reuse many elements of the prompt (BM25 and embedding similarity), which can make them look cleaner and more consistent to the judge.

In conclusion, according to the judge, the generated continuations are at the level of the real stories on grammar, creativity and consistency, but the judge is not discriminative enough to rank the three kinds of text. A larger judge model and more stories would be needed for a finer evaluation.

### 9. Summary

In [45]:
summary = pd.DataFrame(results).T[["real", "greedy", "sampling"]]
print(f"Perplexity of our model: validation {ppl_val:.2f}, training {ppl_train:.2f}")
print(f"Embedding similarity, random baseline: {random_baseline:.3f}")
summary.round(3)

Perplexity of our model: validation 5.30, training 5.07
Embedding similarity, random baseline: 0.173


,real,greedy,sampling
GPT-2 perplexity (median),12.505,9.246,12.280
Similarity prompt / continuation,0.256,0.269,0.264
Similarity generated / real continuation,1.000,0.681,0.662
spaCy error rate (sentences),0.010,0.008,0.015
CYK parse rate,0.713,0.798,0.780
BM25 top-1 accuracy,0.480,0.440,0.440
BM25 MRR,0.580,0.546,0.538
distinct-1,0.688,0.639,0.677
distinct-2,0.948,0.902,0.935
distinct-3,0.989,0.961,0.983


## General conclusion

### 1. What we built

We implemented from scratch a decoder-only transformer following the Llama 2 architecture, trained
it on TinyStories, implemented three decoding methods, and designed an evaluation protocol for the
generated stories. The final model has 8 decoder blocks, an embedding size of 128, 4 attention
heads and a feed-forward intermediate size of 256, for about 11.3 million parameters.

### 2. Architecture choices

Each component follows a choice made by Llama 2, for reasons that have been validated on large
models:

- **Pre-normalization with RMSNorm.** The normalization is applied *before* the attention and the
  feed-forward network, not after. The residual path then carries the signal from the input to the
  output without being normalized at each block, which stabilizes the gradients in deep networks.
  RMSNorm only rescales the vectors by their root mean square, without centering them: it is
  cheaper than LayerNorm and works as well in practice.
- **Residual connections.** Following the standard Llama block, the feed-forward output is added to
  $x_3$ (the residual stream) and not to $x_4$ (its normalized version) as written in the
  statement. Adding the normalized vector would break the residual path, which is precisely what
  pre-normalization is meant to preserve.
- **RoPE positional encoding.** Instead of adding a learned position vector to the embeddings, RoPE
  rotates the queries and keys according to their position. The attention score between two tokens
  then only depends on their *relative* distance, the encoding has no parameters, and it applies
  naturally to the KV cache, since each new token is rotated with its own position.
- **SwiGLU feed-forward network.** The feed-forward network combines a SiLU-activated gate with a
  linear projection ($SiLU(W^g x) \odot W^u x$). This gating mechanism gives better results than a
  classic ReLU or GELU network for a similar number of parameters.
- **Batched multi-head attention with a causal mask.** All heads are computed in a single matrix
  operation instead of a loop, which is much faster on GPU. The causal mask prevents each token from
  attending to future tokens, which is both what allows the model to be trained on all positions of
  a sequence at once and what makes the KV cache possible at generation time.

Our model also differs from Llama 2 on a few points, mostly imposed by the statement or the
skeleton: each head has a dimension equal to the embedding size (128) instead of splitting it
between heads, the linear layers have biases, the final normalization is a LayerNorm, and the
embedding and output matrices are not shared. These differences do not change the nature of the
architecture.

The parameters of our model are unevenly distributed, because of the tokenizer. The embedding
layer stores one vector of 128 numbers for each of the 32,768 tokens of the Mistral vocabulary
(4.2 million parameters), and the output layer computes one score per token from these 128
numbers (4.2 million more). Together, these two layers only convert token ids into vectors and
back, but they contain 74% of the 11.3 million parameters, while the 8 transformer blocks, where
attention and the feed-forward networks actually model the language, only contain 2.9 million.

The Mistral tokenizer was designed for a general-purpose model: its vocabulary covers many
languages, code and symbols, most of which never appear in TinyStories. A tokenizer trained on
TinyStories itself would need a much smaller vocabulary. With 8,192 tokens, for instance, the
embedding and output layers would only contain about 2.1 million parameters: for the same total
budget, the model could have about 25 blocks instead of 8.

### 3. Training

The model was trained for one epoch on the 2.1 million training stories (132,000 iterations with a
batch of 16, stories truncated to 128 tokens), with AdamW and a learning rate of $6 \times 10^{-4}$.
The loss decreased from 10.5, the value of a random guess over the vocabulary ($\ln 32768 \approx
10.4$), to about 1.63, and was still slowly decreasing at the end of the epoch. The perplexity is
5.07 on training stories and 5.30 on validation stories: the small gap shows that the model
generalizes to unseen stories and does not overfit.

### 4. Decoding

The greedy decoding with a KV cache produces exactly the same text as the naive version, which
validates our implementation of the cache, while being faster since the keys and values of the
previous tokens are not recomputed at each step. Greedy decoding produces fluent but repetitive
stories ("She was scared and ran away. [...] Alice was very scared and ran away."). Sampling
diversifies the text, and the temperature controls the trade-off between diversity and coherence:
at $T = 1.2$, the story becomes hard to follow, with invented words and events without logical
links.

### 5. Results of the evaluation

| Aspect | Result |
|---|---|
| Fluency | GPT-2 perplexity of 12.3 for sampling, against 12.5 for the real continuations |
| Coherence with the prompt | similarity and BM25 retrieval at the level of the real continuations |
| Diversity inside a story | distinct-2 of 0.90 (greedy) and 0.93 (sampling), against 0.95 (real) |
| Diversity across stories | corpus distinct-2 of 0.41 (greedy) and 0.53 (sampling), against 0.63 (real) |
| Vocabulary | 576 (greedy) and 753 (sampling) distinct words, against 965 (real) |
| Grammar | no degradation detected (spaCy errors around 1%, CYK parse rate at least as high as real) |
| LLM judge | grades at the level of the real continuations, but the judge is not discriminative |

Taken together, the metrics give a consistent picture. **Locally, the generated stories are at the
level of the real ones**: they are fluent, grammatical, talk about the characters and objects of
the prompt, and contain few repetitions inside a story. **Their weakness is diversity**: the model
reuses the same expressions and a smaller vocabulary from one story to another, especially with
greedy decoding. Several metrics where greedy decoding scores *better* than the real texts (lower
judge perplexity, higher CYK parse rate, higher judge grades) are explained by this lack of
diversity: simple and predictable sentences are favored by these metrics. Sampling with $T = 0.7$
is the best compromise: it is as fluent as the real stories while being clearly more diverse than
greedy decoding.

The evaluation itself has limits: comparing to a single real continuation penalizes correct but
different texts, our grammar only covers part of English, and the small LLM judge cannot separate
texts of similar quality. Using the real continuations as a reference for every metric, and random
baselines for the similarities, was essential to interpret the values.

### 6. Is our transformer as viable as a transformer in general?

**As an architecture, yes.** Our model uses the same components as Llama 2 (pre-normalization with
RMSNorm, RoPE, SwiGLU, causal multi-head attention, KV cache), and the results show that this
architecture works: the model learned to produce grammatical, coherent stories, at the level of the
real ones on most of our metrics, after a single epoch. This is consistent with the observation of
the TinyStories authors that very small models can generate fluent stories when the domain is
restricted.

**As a language model, no**, and the difference is a matter of scale rather than of design. Llama 2
7B has about 600 times more parameters, 32 blocks instead of 8, an embedding size of 4,096 instead
of 128, a context of 4,096 tokens instead of 128, and was trained on 2 trillion tokens, against at
most about 270 million for our model. Our model only knows the world of TinyStories: it has no
general knowledge, cannot follow instructions, cannot handle prompts outside this domain, and its
coherence degrades on long texts, as the story generated at $T = 1.2$ shows. Its weaknesses
(limited diversity across stories, short-range coherence) are those expected from a small model
trained on a small and homogeneous dataset.

Several improvements would bring it closer to a real language model, without changing the
architecture: a tokenizer adapted to the dataset to give more parameters to the transformer blocks,
more layers and a larger embedding size, a longer context, excluding padding tokens from the
training loss, a learning rate schedule with warm-up and decay, and more training data and epochs.
In short, we reproduced a viable transformer architecture at a very small scale: it is a working
miniature of a large language model, not a substitute for one.